In [94]:
SEED = 42

# Data configuration
DATA_ID = 'TIGER-Lab/ViRL39K'
DATA_SPLIT = 'train'
DATA_SIZE = 1250
TEST_RATIO = 0.1

In [47]:
from datasets import load_dataset, Dataset
from collections import deque

def load_hf_dataset(data_id, split, total_size, from_end=False):
    dataset_stream = load_dataset(data_id, split=split, streaming=True)

    if not from_end:
        # Slice from the start
        sliced_data = []
        for example in dataset_stream:
            if len(sliced_data) >= total_size:
                break
            sliced_data.append(example)
    else:
        # Slice from the end using a fixed-size buffer
        buffer = deque(maxlen=total_size)
        for example in dataset_stream:
            buffer.append(example)
        sliced_data = list(buffer)

    dataset = Dataset.from_list(sliced_data)
    return dataset

# Load the RL dataset
dataset = load_hf_dataset(
    DATA_ID, 
    split=DATA_SPLIT,
    total_size=38_869, # Actual size is 38,870, but loading all causes an error, so we load one less than the actual size
    from_end=False,
)

# Add 'category_label' column and encode it
dataset = dataset.add_column('category_label', dataset['category'])
dataset = dataset.class_encode_column('category_label')

# Create 'stratify' column
df = dataset.to_pandas()
df['source_merged'] = df['source'].str.replace('Converted', '')
df['stratify'] = df['source_merged'].astype(str) + ' - ' + df['category'].astype(str)
stratify_list = df['stratify'].unique().tolist()
dataset = Dataset.from_pandas(df, preserve_index=False)

print(dataset)
print("Stratify list:", stratify_list)

Casting to class labels:   0%|          | 0/38869 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image', 'category_label', 'source_merged', 'stratify'],
    num_rows: 38869
})
Stratify list: ['Processed - (GradeSchool) Geometric', 'Processed - Spatial Reasoning', 'Processed - (GradeSchool) Non-Geo Math', 'Processed - Tables/Diagrams/Charts', 'Processed - (GradeSchool) Science', 'Processed - Commonsense', 'Processed - Social Science', 'Processed - Broader STEM Topics', 'R1OneVision - Tables/Diagrams/Charts', 'R1OneVision - (GradeSchool) Science', 'R1OneVision - Broader STEM Topics', 'R1OneVision - (GradeSchool) Non-Geo Math', 'R1OneVision - (GradeSchool) Geometric', 'R1OneVision - Spatial Reasoning', 'MMMath - (GradeSchool) Non-Geo Math', 'DeepScaleR - (GradeSchool) Non-Geo Math', 'geoqa_plus - (GradeSchool) Geometric', 'ScienceQA - (GradeSchool) Science', 'dvqa - Tables/Diagrams/Charts', 'K12 - (GradeSchool) Non-Geo Math', 'M3CoT - Spatial Reasoning', 'M3CoT - (Gr

In [48]:
# Filter out examples with multiple images
dataset_filtered = dataset.filter(lambda x: len(x['image']) == 1)
print(dataset_filtered)

Filter:   0%|          | 0/38869 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image', 'category_label', 'source_merged', 'stratify'],
    num_rows: 36580
})


In [49]:
df_filtered = dataset_filtered.to_pandas()

# Filter out examples with non-numeric answers
df_filtered = df_filtered[df_filtered['answer'].str.contains(r"\d", na=False)]

# Filter out examples with long answers where 'answer_length' > Q3
df_filtered['answer_length'] = df_filtered['answer'].str.len()
answer_length_q3 = df_filtered['answer_length'].quantile(0.75)
df_filtered = df_filtered[df_filtered['answer_length'] <= answer_length_q3]

df_filtered

,question,answer,PassRate_32BTrained,PassRate_7BBase,category,source,qid,image,category_label,source_merged,stratify,answer_length
3,<image>\nHow many cans are there?,\boxed{100},1.0000,1.000,Spatial Reasoning,Processed,Processed-bba2265d-739f-4f64-8a21-039a48cde582,[images/Processed-bba2265d-739f-4f64-8a21-039a...,6,Processed,Processed - Spatial Reasoning,11
4,<image>\nHow many squares are there?,\boxed{100},1.0000,1.000,(GradeSchool) Geometric,Processed,Processed-45a6e75a-124a-4ff6-adda-5b1f518d13af,[images/Processed-45a6e75a-124a-4ff6-adda-5b1f...,0,Processed,Processed - (GradeSchool) Geometric,11
5,<image>\nHow many diamonds are there?,\boxed{20},1.0000,1.000,(GradeSchool) Geometric,Processed,Processed-1cd8a0fe-2bfe-4f8a-99e4-cae1dffa9e96,[images/Processed-1cd8a0fe-2bfe-4f8a-99e4-cae1...,0,Processed,Processed - (GradeSchool) Geometric,10
11,"<image>\nIf you add 3 dots to the second row, ...",\boxed{15},0.9375,-1.000,Spatial Reasoning,Processed,Processed-40ec1a41-7d05-418b-adaf-5e69ecc47638,[images/Processed-40ec1a41-7d05-418b-adaf-5e69...,6,Processed,Processed - Spatial Reasoning,10
13,<image>\nHow many paper clips are partially vi...,\boxed{2},0.5000,-1.000,Spatial Reasoning,Processed,Processed-c0c920a3-423f-4ba0-b8d3-a1a0868ce80e,[images/Processed-c0c920a3-423f-4ba0-b8d3-a1a0...,6,Processed,Processed - Spatial Reasoning,9
...,...,...,...,...,...,...,...,...,...,...,...,...
33882,Find the value of x.\nThe above problem is wit...,\boxed{6},1.0000,0.125,(GradeSchool) Geometric,Processed,Processed-29134474-b0ad-4c79-aa95-7067d1fdb7bc,[images/Processed-ada82124-edff-48c2-b84d-7ae6...,0,Processed,Processed - (GradeSchool) Geometric,9
33883,The below problem is with the following images...,\boxed{1.5},0.2500,-1.000,(GradeSchool) Geometric,Processed,Processed-442ec8b5-a399-4392-aeb4-012b2a58ac02,[images/Processed-ecb571eb-e804-4884-8150-c14c...,0,Processed,Processed - (GradeSchool) Geometric,11
33884,"In the figure, point O is a point inside trian...",\boxed{50},0.8750,-1.000,(GradeSchool) Geometric,Processed,Processed-a5b2b903-1dd5-47c3-bc81-b151cd715600,[images/Processed-2b726995-cb8e-4c36-a231-e0c4...,0,Processed,Processed - (GradeSchool) Geometric,10
33885,The below problem is with the following images...,\boxed{45},0.5000,-1.000,(GradeSchool) Geometric,Processed,Processed-ee8a2e6b-7770-4fcd-9235-855e66625cbc,[images/Processed-ee8a2e6b-7770-4fcd-9235-855e...,0,Processed,Processed - (GradeSchool) Geometric,10


In [54]:
stratify_list_filtered = df_filtered['stratify'].unique().tolist()
stratify_list_got_filtered = list(set(stratify_list) - set(stratify_list_filtered))
print("Stratify list after filtering:", stratify_list_filtered)
print("Stratify list that got filtered:", stratify_list_got_filtered)

Stratify list after filtering: ['Processed - Spatial Reasoning', 'Processed - (GradeSchool) Geometric', 'Processed - (GradeSchool) Non-Geo Math', 'Processed - Tables/Diagrams/Charts', 'Processed - (GradeSchool) Science', 'Processed - Social Science', 'Processed - Commonsense', 'Processed - Broader STEM Topics', 'R1OneVision - (GradeSchool) Science', 'R1OneVision - (GradeSchool) Non-Geo Math', 'R1OneVision - (GradeSchool) Geometric', 'R1OneVision - Spatial Reasoning', 'R1OneVision - Broader STEM Topics', 'MMMath - (GradeSchool) Non-Geo Math', 'DeepScaleR - (GradeSchool) Non-Geo Math', 'geoqa_plus - (GradeSchool) Geometric', 'dvqa - Tables/Diagrams/Charts', 'K12 - (GradeSchool) Non-Geo Math', 'ai2d - Tables/Diagrams/Charts', 'dvqa - Broader STEM Topics', 'chartqa - Tables/Diagrams/Charts', 'dvqa - Social Science', 'MMK12 - Tables/Diagrams/Charts', 'MMK12 - (GradeSchool) Non-Geo Math', 'MMK12 - (GradeSchool) Geometric', 'MMK12 - Spatial Reasoning', 'MMK12 - Broader STEM Topics', 'MMK12 - 

In [ ]:
print("Dataset size before sampling:", len(df_filtered))

stratify_counts = df_filtered['stratify'].value_counts().sort_values(ascending=False)
least_stratify_list = stratify_counts[stratify_counts < 10].index.tolist()
print("Least stratify list:", least_stratify_list)

mask = df_filtered['stratify'].isin(least_stratify_list)
df_no_sample = df_filtered[mask]
df_to_sample = df_filtered[~mask]

print("Dataset size to not sample:", len(df_no_sample))
print("Dataset size to sample:", len(df_to_sample))

Dataset size before sampling: 18625
Least stratify list: ['R1OneVision - Broader STEM Topics', 'Processed - Social Science', 'geoqa_plus - Spatial Reasoning', 'geoqa_plus - (GradeSchool) Non-Geo Math', 'M3CoT - Spatial Reasoning', 'Processed - Commonsense', 'dvqa - Broader STEM Topics', 'dvqa - Social Science', 'M3CoT - Broader STEM Topics', 'ai2d - (GradeSchool) Science']
Dataset size to not sample: 34
Dataset size to sample: 18591


In [ ]:
import numpy as np
from datasets import concatenate_datasets

# Get normalized distribution
df = dataset.to_pandas()
category_counts = df[~df['stratify'].isin(stratify_list_got_filtered + least_stratify_list)]['stratify'].value_counts(normalize=True)

# Raw allocation (float)
raw = category_counts * (DATA_SIZE - len(df_no_sample))

# Floor allocation
samples_per_category = np.floor(raw).astype(int)

# Distribute remainder
remainder = (DATA_SIZE - len(df_no_sample)) - samples_per_category.sum()
fractional = raw - samples_per_category

for cat in fractional.sort_values(ascending=False).index[:remainder]:
    samples_per_category[cat] += 1

# Stratified sampling
dataset_to_sample = Dataset.from_pandas(df_to_sample, preserve_index=False)
sampled_splits = []
global_diff = 0
for category, n in samples_per_category.sort_values(ascending=True).items():
    subset = dataset_to_sample.filter(lambda x: x['stratify'] == category)
    n_population = len(subset)
    subset = subset.shuffle(seed=SEED).select(range(min(n + global_diff, len(subset))))
    
    # Update global_diff to compensate for any sampling deficit in the next category
    diff = n - len(subset)
    global_diff = diff if diff > 0 else 0
    print(f"Category {category}: n_population={n_population}, n_target={n}, n_sampled={len(subset)}, global_diff={global_diff}\n")
    
    sampled_splits.append(subset)

# Combine all
dataset_sampled = concatenate_datasets(sampled_splits)
print(dataset_sampled)

Filter:   0%|          | 0/18591 [00:00<?, ? examples/s]

Category R1OneVision - Tables/Diagrams/Charts: n_population=10, n_target=1, n_sampled=1, global_diff=0



Filter:   0%|          | 0/18591 [00:00<?, ? examples/s]

Category MMK12 - Social Science: n_population=10, n_target=1, n_sampled=1, global_diff=0



Filter:   0%|          | 0/18591 [00:00<?, ? examples/s]

...

Category MMMath - (GradeSchool) Non-Geo Math: n_population=2522, n_target=113, n_sampled=113, global_diff=0



Filter:   0%|          | 0/18591 [00:00<?, ? examples/s]

Category MMK12 - (GradeSchool) Geometric: n_population=3336, n_target=173, n_sampled=173, global_diff=0



Filter:   0%|          | 0/18591 [00:00<?, ? examples/s]

Category Processed - (GradeSchool) Non-Geo Math: n_population=1274, n_target=177, n_sampled=177, global_diff=0

Dataset({
    features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image', 'category_label', 'source_merged', 'stratify', 'answer_length'],
    num_rows: 1216
})


In [79]:
# Download the images.zip file
from huggingface_hub import hf_hub_download

images_zip_path = hf_hub_download(
    repo_id=DATA_ID,
    filename='images.zip',
    repo_type='dataset',
)
print(images_zip_path)

/root/.cache/huggingface/hub/datasets--TIGER-Lab--ViRL39K/snapshots/812ec617dea4bc8a4e751663b88e4ebb7de4d00e/images.zip


In [80]:
# Extract the images.zip file
import os
if not os.path.isdir('images'):
    import zipfile
    with zipfile.ZipFile(images_zip_path, 'r') as zip_ref:
        zip_ref.extractall()

In [119]:
# Check category distribution in the processed dataset
import pandas as pd

df = pd.concat([dataset_sampled.to_pandas(), df_no_sample])
print("Dataset size:", len(df))
print(df['stratify'].value_counts())

Dataset size: 1250
stratify
Processed - (GradeSchool) Non-Geo Math      177
MMK12 - (GradeSchool) Geometric             173
MMMath - (GradeSchool) Non-Geo Math         113
Processed - (GradeSchool) Geometric         112
MMK12 - (GradeSchool) Non-Geo Math          104
Processed - Tables/Diagrams/Charts           76
MMK12 - Tables/Diagrams/Charts               54
geoqa_plus - (GradeSchool) Geometric         51
MMK12 - (GradeSchool) Science                46
DeepScaleR - (GradeSchool) Non-Geo Math      44
K12 - (GradeSchool) Non-Geo Math             36
dvqa - Tables/Diagrams/Charts                35
MMK12 - Spatial Reasoning                    26
ai2d - Tables/Diagrams/Charts                23
Processed - Spatial Reasoning                21
M3CoT - (GradeSchool) Science                20
Processed - (GradeSchool) Science            16
ScienceQA - (GradeSchool) Science            16
Processed - Broader STEM Topics              14
M3CoT - Social Science                       14
R1OneVision 

In [ ]:
# Exclude samples from the smallest category before stratification
# smallest_cat_count = cat_counts.iloc[-1]
# smallest_cat_label = cat_counts.index[-1][1]

# drop_idxs = df[df['category_label'] == smallest_cat_label].sample(smallest_cat_count, random_state=SEED).index
# to_stratify_df = df.drop(index=drop_idxs)
# no_stratify_df = df.loc[drop_idxs]

# print("Number of samples to stratify:", len(to_stratify_df))
# print("Number of samples not to stratify:", len(no_stratify_df))

In [120]:
df['stratify'] = df['stratify'].replace({
    'R1OneVision - Spatial Reasoning': 'geoqa_plus - Spatial Reasoning',
    'R1OneVision - (GradeSchool) Geometric': 'geoqa_plus - (GradeSchool) Geometric',
    'M3CoT - Spatial Reasoning': 'geoqa_plus - Spatial Reasoning',
    'geoqa_plus - (GradeSchool) Non-Geo Math': 'K12 - (GradeSchool) Non-Geo Math',
    'M3CoT - Tables/Diagrams/Charts': 'ai2d - Tables/Diagrams/Charts',
    'dvqa - Broader STEM Topics': 'R1OneVision - Broader STEM Topics',
    'R1OneVision - (GradeSchool) Non-Geo Math': 'K12 - (GradeSchool) Non-Geo Math',
    'MMK12 - Social Science': 'M3CoT - Social Science',
    'R1OneVision - Tables/Diagrams/Charts': 'ai2d - Tables/Diagrams/Charts',
    'dvqa - Social Science': 'M3CoT - Social Science',
    'M3CoT - Broader STEM Topics': 'R1OneVision - Broader STEM Topics',
    'ai2d - (GradeSchool) Science': 'R1OneVision - (GradeSchool) Science',
    'chartqa - Tables/Diagrams/Charts': 'ai2d - Tables/Diagrams/Charts',
    'Processed - Social Science': 'M3CoT - Social Science',
})
stratify_counts = df['stratify'].value_counts()
print(stratify_counts)

stratify
Processed - (GradeSchool) Non-Geo Math     177
MMK12 - (GradeSchool) Geometric            173
MMMath - (GradeSchool) Non-Geo Math        113
Processed - (GradeSchool) Geometric        112
MMK12 - (GradeSchool) Non-Geo Math         104
Processed - Tables/Diagrams/Charts          76
MMK12 - Tables/Diagrams/Charts              54
geoqa_plus - (GradeSchool) Geometric        54
MMK12 - (GradeSchool) Science               46
DeepScaleR - (GradeSchool) Non-Geo Math     44
K12 - (GradeSchool) Non-Geo Math            41
dvqa - Tables/Diagrams/Charts               35
ai2d - Tables/Diagrams/Charts               32
MMK12 - Spatial Reasoning                   26
M3CoT - Social Science                      23
Processed - Spatial Reasoning               21
M3CoT - (GradeSchool) Science               20
Processed - (GradeSchool) Science           16
ScienceQA - (GradeSchool) Science           16
R1OneVision - (GradeSchool) Science         14
Processed - Broader STEM Topics             14
MMK1

In [121]:
least_stratify_list = stratify_counts[stratify_counts < 3].index.tolist()
df_no_stratify = df[df['stratify'].isin(least_stratify_list)]
df_to_stratify = df[~df['stratify'].isin(least_stratify_list)]

print("Number of samples to stratify:", len(df_to_stratify))
print("Number of samples not to stratify:", len(df_no_stratify))

Number of samples to stratify: 1248
Number of samples not to stratify: 2


In [139]:
# Stratified split the dataset into train, validation, and test sets
from sklearn.model_selection import train_test_split
import pandas as pd
from datasets import DatasetDict

train_df, val_test_df = train_test_split(
    df_to_stratify,
    test_size=(int(DATA_SIZE*TEST_RATIO*2) - len(df_no_stratify)),
    stratify=df_to_stratify['stratify'],
    random_state=SEED,
)
val_test_df = pd.concat([val_test_df, df_no_stratify], ignore_index=True)
val_df, test_df = train_test_split(
    val_test_df,
    test_size=0.5,
    stratify=val_test_df['stratify'],
    random_state=SEED,
)

print("Train set size:", len(train_df))
print("Validation set size:", len(val_df))
print("Test set size:", len(test_df))

Train set size: 1000
Validation set size: 125
Test set size: 125


In [128]:
# # Manually fix the split set sizes
# test_df_sampled = test_df[test_df['category_label'] == 0].sample(1, random_state=SEED)
# train_df = pd.concat([train_df, test_df_sampled], ignore_index=True)
# test_df = test_df.drop(index=test_df_sampled.index)

# print("Train set size:", len(train_df))
# print("Validation set size:", len(val_df))
# print("Test set size:", len(test_df))

In [142]:
selected_cols = ['qid', 'image', 'question', 'answer', 'category', 'source', 'PassRate_32BTrained', 'PassRate_7BBase']
train_set = Dataset.from_pandas(train_df[selected_cols], preserve_index=False)
val_set = Dataset.from_pandas(val_df[selected_cols], preserve_index=False)
test_set = Dataset.from_pandas(test_df[selected_cols], preserve_index=False)

print("Train set:")
print(train_set)
print()
print("Validation set:")
print(val_set)
print()
print("Test set:")
print(test_set)

Train set:
Dataset({
    features: ['qid', 'image', 'question', 'answer', 'category', 'source', 'PassRate_32BTrained', 'PassRate_7BBase'],
    num_rows: 1000
})

Validation set:
Dataset({
    features: ['qid', 'image', 'question', 'answer', 'category', 'source', 'PassRate_32BTrained', 'PassRate_7BBase'],
    num_rows: 125
})

Test set:
Dataset({
    features: ['qid', 'image', 'question', 'answer', 'category', 'source', 'PassRate_32BTrained', 'PassRate_7BBase'],
    num_rows: 125
})


In [143]:
from PIL import Image

def process_image_with_path(example):
    image_paths = example['image']
    assert len(image_paths) == 1, "Expected exactly one image path per example!"
    image_path = image_paths[0]
    image = Image.open(image_path)
    example['PIL_image'] = image
    return example

# Process the split sets to load images
train_set = train_set.map(process_image_with_path, num_proc=4)
val_set = val_set.map(process_image_with_path, num_proc=4)
test_set = test_set.map(process_image_with_path, num_proc=4)

# Rename columns for clarity
rename_cols_map = {'image': 'image_paths', 'PIL_image': 'image'}
selected_cols = ['qid', 'image', 'question', 'answer', 'category', 'source', 'PassRate_32BTrained', 'PassRate_7BBase']
train_set = train_set.rename_columns(rename_cols_map)
val_set = val_set.rename_columns(rename_cols_map)
test_set = test_set.rename_columns(rename_cols_map)

assert (
    isinstance(train_set[0]['image'], Image.Image) and
    isinstance(val_set[0]['image'], Image.Image) and
    isinstance(test_set[0]['image'], Image.Image)
), "Expected 'image' column to contain PIL Image objects!"

print("Train set:")
print(train_set)
print()
print("Validation set:")
print(val_set)
print()
print("Test set:")
print(test_set)

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/125 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/125 [00:00<?, ? examples/s]

Train set:
Dataset({
    features: ['qid', 'image_paths', 'question', 'answer', 'category', 'source', 'PassRate_32BTrained', 'PassRate_7BBase', 'image'],
    num_rows: 1000
})

Validation set:
Dataset({
    features: ['qid', 'image_paths', 'question', 'answer', 'category', 'source', 'PassRate_32BTrained', 'PassRate_7BBase', 'image'],
    num_rows: 125
})

Test set:
Dataset({
    features: ['qid', 'image_paths', 'question', 'answer', 'category', 'source', 'PassRate_32BTrained', 'PassRate_7BBase', 'image'],
    num_rows: 125
})


In [145]:
# Create a DatasetDict for the splits
dataset_split = DatasetDict({
    'train': train_set.select_columns(selected_cols),
    'val': val_set.select_columns(selected_cols),
    'test': test_set.select_columns(selected_cols),
})
print(dataset_split)

DatasetDict({
    train: Dataset({
        features: ['qid', 'image', 'question', 'answer', 'category', 'source', 'PassRate_32BTrained', 'PassRate_7BBase'],
        num_rows: 1000
    })
    val: Dataset({
        features: ['qid', 'image', 'question', 'answer', 'category', 'source', 'PassRate_32BTrained', 'PassRate_7BBase'],
        num_rows: 125
    })
    test: Dataset({
        features: ['qid', 'image', 'question', 'answer', 'category', 'source', 'PassRate_32BTrained', 'PassRate_7BBase'],
        num_rows: 125
    })
})


In [146]:
# Push the dataset to Hugging Face Hub
dataset_split.push_to_hub(f'alxxtexxr/ViRL{DATA_SIZE/1000}K')

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  95%|#########4| 4.83MB / 5.09MB            

README.md:   0%|          | 0.00/787 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/alxxtexxr/ViRL1.25K/commit/8f88d7ba9d5ec1ef3152a134bd7c94be2479610a', commit_message='Upload dataset', commit_description='', oid='8f88d7ba9d5ec1ef3152a134bd7c94be2479610a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alxxtexxr/ViRL1.25K', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alxxtexxr/ViRL1.25K'), pr_revision=None, pr_num=None)